'answer_classifier.ipynb'는 실행할 필요가 없으며 해당 README.md를 읽어주세요

In [4]:
"""
두 번째 코드: Answer Classifier with Advanced Models
- 3가지 모델 base model: EfficientNet, ResNet, MobileNet
- Answer_1, Answer_2 데이터셋 추가 파인튜닝
"""

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import logging
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import StratifiedKFold
import numpy as np
from huggingface_hub import hf_hub_download
import os
import copy 

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# ========== Model Architectures (첫 번째 코드와 동일하게 설정) ==========

class SEBlock(nn.Module):
    """Squeeze-and-Excitation block"""
    def __init__(self, in_channels, se_channels):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, se_channels, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(se_channels, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ResidualBlock(nn.Module):
    """Residual connection wrapper"""
    def __init__(self, module):
        super().__init__()
        self.module = module

    def forward(self, x):
        return x + self.module(x)


class BasicBlock(nn.Module):
    """Basic block for ResNet"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class InvertedResidual(nn.Module):
    """Inverted Residual block for MobileNetV3"""
    def __init__(self, in_channels, out_channels, kernel, stride, expand_ratio, se_ratio=None):
        super().__init__()

        hidden_dim = int(in_channels * expand_ratio)
        self.use_res_connect = stride == 1 and in_channels == out_channels

        layers = []

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.Hardswish(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.Hardswish(inplace=True)
        ])

        if se_ratio is not None:
            se_channels = int(in_channels * se_ratio)
            layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class EfficientNetB0_MNIST(nn.Module):
    """EfficientNet-B0 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(32)
        self.act1 = nn.SiLU(inplace=True)

        self.blocks = nn.Sequential(
            self._make_mbconv(32, 16, kernel=3, stride=1, expand_ratio=1),
            self._make_mbconv(16, 24, kernel=3, stride=2, expand_ratio=6),
            self._make_mbconv(24, 24, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(24, 40, kernel=5, stride=2, expand_ratio=6),
            self._make_mbconv(40, 40, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(40, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 80, kernel=3, stride=1, expand_ratio=6),
            self._make_mbconv(80, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 112, kernel=5, stride=1, expand_ratio=6),
            self._make_mbconv(112, 192, kernel=5, stride=1, expand_ratio=6),
        )

        self.conv_head = nn.Conv2d(192, 1280, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(1280)
        self.act2 = nn.SiLU(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )

    def _make_mbconv(self, in_channels, out_channels, kernel, stride, expand_ratio):
        layers = []
        hidden_dim = in_channels * expand_ratio

        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])

        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel, stride,
                     padding=kernel//2, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True)
        ])

        se_channels = max(1, in_channels // 4)
        layers.append(SEBlock(hidden_dim, se_channels))

        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        if stride == 1 and in_channels == out_channels:
            return ResidualBlock(nn.Sequential(*layers))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


class ResNet18_MNIST(nn.Module):
    """ResNet-18 for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_channels, out_channels, blocks, stride=1):
        layers = []
        layers.append(BasicBlock(in_channels, out_channels, stride))
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


class MobileNetV3_MNIST(nn.Module):
    """MobileNetV3-Small for MNIST"""
    def __init__(self, num_classes=10):
        super().__init__()

        self.conv_stem = nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.act1 = nn.Hardswish(inplace=True)

        self.blocks = nn.Sequential(
            InvertedResidual(16, 16, kernel=3, stride=1, expand_ratio=1, se_ratio=0.25),
            InvertedResidual(16, 24, kernel=3, stride=2, expand_ratio=4.5, se_ratio=None),
            InvertedResidual(24, 24, kernel=3, stride=1, expand_ratio=3.67, se_ratio=None),
            InvertedResidual(24, 40, kernel=5, stride=2, expand_ratio=4, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 40, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(40, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 48, kernel=5, stride=1, expand_ratio=3, se_ratio=0.25),
            InvertedResidual(48, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
            InvertedResidual(96, 96, kernel=5, stride=1, expand_ratio=6, se_ratio=0.25),
        )

        self.conv_head = nn.Conv2d(96, 576, kernel_size=1, bias=False)
        self.bn2 = nn.BatchNorm2d(576)
        self.act2 = nn.Hardswish(inplace=True)

        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(576, 1024),
            nn.Hardswish(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.conv_stem(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.blocks(x)
        x = self.conv_head(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# ========== Dataset Class ==========

class AnswerDataset(Dataset):
    """Answer_1, Answer_2 데이터셋 + 증강"""

    def __init__(self, image_paths, labels, augment=False):
        self.image_paths = image_paths
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(image_path).convert('L')
        image = image.resize((28, 28))

        # Tensor 변환
        image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0

        # 데이터 증강 (Train만)
        if self.augment:
            # 랜덤 회전
            if torch.rand(1) > 0.5:
                angle = torch.randint(-10, 11, (1,)).item()
                image = transforms.functional.rotate(
                    transforms.ToPILImage()(image.squeeze(0)), angle
                )
                image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0

            # 수직/수평 이동
            if torch.rand(1) > 0.5:
                tx = torch.randint(-2, 3, (1,)).item() # 수평 이동 -2 ~ +2 픽셀
                ty = torch.randint(-2, 3, (1,)).item() # 수직 이동 -2 ~ +2 픽셀
                image = transforms.functional.affine(
                    transforms.ToPILImage()(image.squeeze(0)),
                    angle=0, translate=(tx, ty), scale=1.0, shear=0
                )
                image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0

            # 밝기/대비 조절
            if torch.rand(1) > 0.5:
                brightness_factor = torch.FloatTensor(1).uniform_(0.8, 1.2).item()
                image = transforms.functional.adjust_brightness(
                    transforms.ToPILImage()(image.squeeze(0)), brightness_factor
                )
                image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0

            # 가우시안 노이즈
            if torch.rand(1) > 0.5:
                noise = torch.randn(image.size()) * 0.02
                image = image + noise
                image = torch.clamp(image, 0.0, 1.0)

            # 가우시안 블러 추가
            if torch.rand(1) > 0.5:
                sigma = torch.FloatTensor(1).uniform_(0.1, 1.0).item()
                image = transforms.functional.gaussian_blur(
                    transforms.ToPILImage()(image.squeeze(0)), kernel_size=3, sigma=sigma
                )
                image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0

            # Affine Shear (왜곡)
            if torch.rand(1) > 0.5:
                shear_factor = torch.FloatTensor(1).uniform_(-5, 5).item() # -5 ~ +5도
                image = transforms.functional.affine(
                    transforms.ToPILImage()(image.squeeze(0)),
                    angle=0, translate=(0, 0), scale=1.0, shear=shear_factor
                )
                image = torch.FloatTensor(np.array(image)).unsqueeze(0) / 255.0


        # MNIST 정규화
        image = (image - 0.1307) / 0.3081

        return image, torch.tensor(label, dtype=torch.long)

MNIST_MODEL_ROOT_DIR = Path("mnist_models")

def load_pretrained(model_type, num_classes_new=5, device='cpu'):
    """사전학습 모델 로드 및 수정"""

    filename = f"best_{model_type}_mnist_model.pth"
    model_path = MNIST_MODEL_ROOT_DIR / filename
    
    if not model_path.exists():
        raise FileNotFoundError(f"사전 학습된 모델 파일을 찾을 수 없습니다: {model_path}")
    
    print(f"Loading pretrained model from: {model_path}")

    # 1. 모델 아키텍처 정의
    if model_type == 'efficientnet':
        model = EfficientNetB0_MNIST(num_classes=10) # 10 클래스로 정의 후 가중치 로드
    elif model_type == 'resnet':
        model = ResNet18_MNIST(num_classes=10)
    elif model_type == 'mobilenet':
        model = MobileNetV3_MNIST(num_classes=10)
    else:
        raise ValueError(f"Unknown model type: {model_type}")
    
    # 2. 가중치 로드
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded pretrained {model_type} model.")
    print(f" MNIST Test Acc: {checkpoint.get('test_acc', 'N/A'):.2f}%")
    
    # 3. 마지막 레이어 5 클래스로 변경
    if model_type == 'efficientnet':
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes_new)
    elif model_type == 'resnet': 
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes_new)
    elif model_type == 'mobilenet':
        in_features = model.classifier[3].in_features
        model.classifier[3] = nn.Linear(in_features, num_classes_new)

    model = model.to(device)
    print(f"Output layer를 {num_classes_new} 클래스로 변경")

    return model

# ========== Training Functions ==========

def load_answer_data_for_cv(data_dir):
    """Answer 데이터 로드: Cross-validation 용"""
    image_paths = []
    labels = []

    for class_idx in range(1, 6):
        class_dir = data_dir / str(class_idx)
        if not class_dir.exists():
            print(f"클래스 디렉토리 없음: {class_dir}")
            continue

        class_images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png"))
        image_paths.extend(class_images)
        labels.extend([class_idx - 1] * len(class_images))

        print(f"  클래스 {class_idx}: {len(class_images)}개 이미지")

    if len(image_paths) == 0:
        raise ValueError(f"데이터를 찾을 수 없습니다: {data_dir}")

    print(f"  총 {len(image_paths)}개 이미지 로드")

    return np.array(image_paths), np.array(labels)


def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """1 에포크 학습"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(train_loader)
    accuracy = 100. * correct / total

    return avg_loss, accuracy


def evaluate(model, val_loader, criterion, device):
    """모델 평가"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    avg_loss = total_loss / len(val_loader)
    accuracy = 100. * correct / total

    return avg_loss, accuracy


def evaluate_detailed(model, val_loader, device):
    """상세 평가 지표"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    cm = confusion_matrix(all_labels, all_preds)

    return {
        'accuracy': accuracy * 100,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'confusion_matrix': cm
    }


def plot_confusion_matrix(cm, model_type, answer_type, fold_idx, save_path):
    """Confusion Matrix 시각화"""
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['1', '2', '3', '4', '5'],
               yticklabels=['1', '2', '3', '4', '5'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{model_type.upper()} - {answer_type} Fold {fold_idx+1} Confusion Matrix')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Confusion Matrix 저장: {save_path}")


# ========== Fine-tuning Function ==========

def finetune_answer_model_with_cv(answer_type, model_type, data_dir, output_dir,
                                 num_epochs=30, learning_rate=0.0005,
                                 use_augmentation=True, early_stopping_patience=10,
                                 n_splits=5):
    """Answer 모델 Fine-tuning"""

    logger.info(f"\n{'='*60}")
    print(f"{answer_type} Fine-tuning with {model_type.upper()} (K={n_splits} Cross Validation)")
    logger.info(f"{'='*60}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info(f"Device: {device}")

    all_image_paths, all_labels = load_answer_data_for_cv(data_dir)

    # Stratified K-Fold for balanced splits
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_results = []
    best_overall_acc = 0.0
    best_overall_model_state_dict = None 

    for fold_idx, (train_index, val_index) in enumerate(skf.split(all_image_paths, all_labels)):
        print(f"\n{'#'*40}")
        print(f"Starting Fold {fold_idx+1}/{n_splits}")
        print(f"{'#'*40}")

        # Create train and validation datasets for this fold
        train_paths, val_paths = all_image_paths[train_index], all_image_paths[val_index]
        train_labels, val_labels = all_labels[train_index], all_labels[val_index]

        train_dataset = AnswerDataset(train_paths, train_labels, augment=use_augmentation)
        val_dataset = AnswerDataset(val_paths, val_labels, augment=False)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

        print(f"  Fold {fold_idx+1}: Train samples = {len(train_paths)}, Validation samples = {len(val_paths)}")
        if use_augmentation:
            print("  Data augmentation enabled for training.")


        # 1. 모델 로드 및 수정
        model = load_pretrained(model_type, num_classes_new=5, device=device)

        # 2. 학습 설정
        criterion = nn.CrossEntropyLoss() 

        # 모델별 최적화 설정
        if model_type == 'efficientnet':
            optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        else:
            optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='max', factor=0.5, patience=3
        )

        # 3. 학습 및 검증 루프
        best_fold_acc = 0.0
        patience_counter = 0
        fold_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        best_model_state_dict_this_fold = None 

        for epoch in range(num_epochs):
            print(f"\n{'='*40}") # Reduce verbosity during CV epochs
            print(f"Fold {fold_idx+1} - Epoch {epoch+1}/{num_epochs}")

            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            val_loss, val_acc = evaluate(model, val_loader, criterion, device)

            fold_history['train_loss'].append(train_loss)
            fold_history['train_acc'].append(train_acc)
            fold_history['val_loss'].append(val_loss)
            fold_history['val_acc'].append(val_acc)

            print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

            scheduler.step(val_acc)

            # Best model saving for this fold
            if val_acc > best_fold_acc:
                best_fold_acc = val_acc
                patience_counter = 0
                best_model_state_dict_this_fold = copy.deepcopy(model.state_dict()) # Deep copy the state dict

                # Optionally save the best model for this fold
                save_dir = Path(output_dir) / f"{answer_type}_{model_type}" / f"fold_{fold_idx+1}"
                save_dir.mkdir(parents=True, exist_ok=True)
                save_path = save_dir / "best_model.pth"
                torch.save(model.state_dict(), save_path)
                print(f"  🎯 Best model for Fold {fold_idx+1} saved! Val Acc: {val_acc:.2f}%")


            else:
                patience_counter += 1
                print(f"  Early Stopping Counter: {patience_counter}/{early_stopping_patience}")

                if patience_counter >= early_stopping_patience:
                    print(f"  Early Stopping at epoch {epoch+1} for Fold {fold_idx+1}")
                    break

        # End of Epoch loop for a fold

        print(f"\nFold {fold_idx+1} finished. Best Validation Accuracy: {best_fold_acc:.2f}%")

        # Evaluate the best model of this fold on its validation set for detailed metrics
        if best_model_state_dict_this_fold is not None:
             model.load_state_dict(best_model_state_dict_this_fold) 
             metrics = evaluate_detailed(model, val_loader, device)
             print(f"  Fold {fold_idx+1} Detailed Metrics:")
             print(f"    Accuracy: {metrics['accuracy']:.2f}%")
             print(f"    Precision: {metrics['precision']:.4f}")
             print(f"    Recall: {metrics['recall']:.4f}")
             print(f"    F1-Score: {metrics['f1_score']:.4f}")

             # Plot Confusion Matrix for this fold
             cm_save_dir = Path(output_dir) / f"{answer_type}_{model_type}" / "confusion_matrices"
             cm_save_dir.mkdir(parents=True, exist_ok=True)
             cm_path = cm_save_dir / f"fold_{fold_idx+1}_confusion_matrix.png"
             plot_confusion_matrix(metrics['confusion_matrix'], model_type, answer_type, fold_idx, cm_path)

             fold_results.append(metrics['accuracy'])

             # Track the best model across all folds
             if best_fold_acc > best_overall_acc:
                 best_overall_acc = best_fold_acc
                 best_overall_model_state_dict = copy.deepcopy(best_model_state_dict_this_fold) # Store the state dict

    # End of Fold loop

    # 4. 결과 집계 및 분석
    print(f"\n{'='*60}")
    print(f"{answer_type} / {model_type.upper()} Cross Validation Results (K={n_splits})")
    print(f"{'='*60}")

    if fold_results:
        mean_accuracy = np.mean(fold_results)
        std_accuracy = np.std(fold_results)
        print(f"Average Validation Accuracy: {mean_accuracy:.2f}% ± {std_accuracy:.2f}%")
        print(f"Individual Fold Accuracies: {fold_results}")

        # Save the model with the highest validation accuracy across all folds
        if best_overall_model_state_dict is not None:
            save_dir = Path(output_dir) / f"{answer_type}_{model_type}"
            save_dir.mkdir(parents=True, exist_ok=True)
            save_path = save_dir / "best_model_overall_cv.pth" # Save with a distinct name

            torch.save({
                'model_type': model_type,
                'model_state_dict': best_overall_model_state_dict,
                'best_cv_acc': best_overall_acc,
                'mean_cv_acc': mean_accuracy,
                'std_cv_acc': std_accuracy
            }, save_path)

            print(f"\n🏆 Best model across all folds saved! Validation Acc: {best_overall_acc:.2f}%")
            print(f"  Model saved to: {save_path}")

    else:
        print("No fold results available.")


    print(f"\n{model_type.upper()} - {answer_type} Cross Validation 완료")


    # Return average accuracy for overall summary
    return mean_accuracy if fold_results else 0.0


# ========== Main Execution ==========

def main():
    """메인 실행 - 지정된 모델로 Answer_1, Answer_2 학습"""

    # 경로 설정
    base_dir = Path(".")

    answer_data_root = base_dir
    output_root = base_dir / "answer_models_cv_results" 

    # 실행할 모델 및 데이터셋 조합
    models_to_run_cv = [
        {'answer': 'answer_1', 'type': 'resnet'}, 
        {'answer': 'answer_2', 'type': 'resnet'},
        {'answer': 'answer_1', 'type': 'mobilenet'},
        {'answer': 'answer_2', 'type': 'mobilenet'},
        {'answer': 'answer_1', 'type': 'efficientnet'},
        {'answer': 'answer_2', 'type': 'efficientnet'},
    ]

    # fold 수
    N_SPLITS = 5

    # 결과 저장용
    cv_results_summary = {}

    for item in models_to_run_cv:
        answer_type = item['answer']
        model_type = item['type']
        data_dir = answer_data_root / answer_type

        if not data_dir.exists():
            print(f"데이터 폴더 없음: {data_dir}. {answer_type}")
            continue

        # Determine appropriate epochs and patience based on previous runs or knowledge
        if answer_type == 'answer_1':
            num_epochs = 60 
            learning_rate = 0.0001
            early_stopping_patience = 20
        elif answer_type == 'answer_2':
            num_epochs = 25
            learning_rate = 0.0005
            early_stopping_patience = 15
        else:
            num_epochs = 30
            learning_rate = 0.0005
            early_stopping_patience = 10


        mean_acc = finetune_answer_model_with_cv(
            answer_type=answer_type,
            model_type=model_type,
            data_dir=data_dir,
            output_dir=output_root,
            num_epochs=num_epochs,
            learning_rate=learning_rate,
            use_augmentation=True,
            early_stopping_patience=early_stopping_patience,
            n_splits=N_SPLITS
        )
        cv_results_summary[f"{answer_type}_{model_type}"] = mean_acc


    # ========== 최종 결과 요약 ==========
    print(f"\n{'='*60}")
    print("🏆 Cross Validation 최종 결과 요약")
    print(f"{'='*60}")

    for answer_type in ['answer_1', 'answer_2']:
        print(f"\n {answer_type} 평균 Validation Accuracy:")
        for model_type in ['efficientnet', 'resnet', 'mobilenet']:
             key = f"{answer_type}_{model_type}"
             if key in cv_results_summary:
                 print(f"  {model_type.upper()}: {cv_results_summary[key]:.2f}%")


    print(f"\n{'='*60}")
    print("모든 Cross Validation 학습 완료")
    print(f"결과 저장 위치: {output_root}")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

2025-10-19 12:04:17,694 - INFO - 
2025-10-19 12:04:17,695 - INFO - ============================================================
2025-10-19 12:04:17,695 - INFO - Device: cpu


answer_1 Fine-tuning with RESNET (K=5 Cross Validation)
  클래스 1: 189개 이미지
  클래스 2: 208개 이미지
  클래스 3: 193개 이미지
  클래스 4: 214개 이미지
  클래스 5: 228개 이미지
  총 1032개 이미지 로드

########################################
Starting Fold 1/5
########################################
  Fold 1: Train samples = 825, Validation samples = 207
  Data augmentation enabled for training.
Loading pretrained model from: mnist_models/best_resnet_mnist_model.pth
Loaded pretrained resnet model.
 MNIST Test Acc: 99.69%
Output layer를 5 클래스로 변경

Fold 1 - Epoch 1/60


  Train Loss: 1.6043, Train Acc: 25.94%
  Val Loss: 1.5219, Val Acc: 30.92%
  🎯 Best model for Fold 1 saved! Val Acc: 30.92%

Fold 1 - Epoch 2/60


  Train Loss: 1.4694, Train Acc: 38.42%
  Val Loss: 1.3765, Val Acc: 43.00%
  🎯 Best model for Fold 1 saved! Val Acc: 43.00%

Fold 1 - Epoch 3/60


  Train Loss: 1.3494, Train Acc: 43.15%
  Val Loss: 1.1385, Val Acc: 56.52%
  🎯 Best model for Fold 1 saved! Val Acc: 56.52%

Fold 1 - Epoch 4/60


...

  Train Loss: 0.0256, Train Acc: 99.19%
  Val Loss: 0.0385, Val Acc: 98.92%
  Early Stopping Counter: 11/15

Fold 5 - Epoch 20/25


  Train Loss: 0.0264, Train Acc: 99.06%
  Val Loss: 0.0419, Val Acc: 98.92%
  Early Stopping Counter: 12/15

Fold 5 - Epoch 21/25


  Train Loss: 0.0477, Train Acc: 98.79%
  Val Loss: 0.0427, Val Acc: 98.38%
  Early Stopping Counter: 13/15

Fold 5 - Epoch 22/25


  Train Loss: 0.0521, Train Acc: 98.38%
  Val Loss: 0.0403, Val Acc: 97.84%
  Early Stopping Counter: 14/15

Fold 5 - Epoch 23/25


  Train Loss: 0.0427, Train Acc: 98.65%
  Val Loss: 0.0313, Val Acc: 99.46%
  Early Stopping Counter: 15/15
  Early Stopping at epoch 23 for Fold 5

Fold 5 finished. Best Validation Accuracy: 99.46%
  Fold 5 Detailed Metrics:
    Accuracy: 99.46%
    Precision: 0.9947
    Recall: 0.9946
    F1-Score: 0.9946
  Confusion Matrix 저장: answer_models_cv_results/answer_2_efficientnet/confusion_matrices/fold_5_confusion_matrix.png

answer_2 / EFFICIENTNET Cross Validation Results (K=5)
Average Validation Accuracy: 99.35% ± 0.63%
Individual Fold Accuracies: [98.9247311827957, 100.0, 100.0, 98.37837837837839, 99.45945945945947]

🏆 Best model across all folds saved! Validation Acc: 100.00%
  Model saved to: answer_models_cv_results/answer_2_efficientnet/best_model_overall_cv.pth

EFFICIENTNET - answer_2 Cross Validation 완료

🏆 Cross Validation 최종 결과 요약

 answer_1 평균 Validation Accuracy:
  EFFICIENTNET: 81.88%
  RESNET: 92.44%
  MOBILENET: 78.00%

 answer_2 평균 Validation Accuracy:
  EFFICIENTNET: 99

Kaggle에 모델 업로드

In [1]:
"""
Fine-tuned 모델을 Kaggle 업로드용으로 준비하는 스크립트
"""

import shutil
from pathlib import Path

def prepare_finetuned_models():
    """Fine-tuned 모델 파일을 업로드 폴더로 복사"""
    
    print("="*70)
    print("📦 Fine-tuned 모델 업로드 준비")
    print("="*70)
    
    # 소스 디렉토리
    source_dir = Path("answer_models_cv_results")
    
    # 업로드 폴더 생성
    upload_dir = Path("kaggle_finetuned_upload")
    if upload_dir.exists():
        shutil.rmtree(upload_dir)
    upload_dir.mkdir()
    
    # 모델 설정
    models = [
        ('answer_1', 'efficientnet'),
        ('answer_1', 'resnet'),
        ('answer_1', 'mobilenet'),
        ('answer_2', 'efficientnet'),
        ('answer_2', 'resnet'),
        ('answer_2', 'mobilenet'),
    ]
    
    print("\n📋 복사할 모델:")
    copied_files = []
    
    for answer_type, model_type in models:
        # 소스 파일 경로
        source_file = source_dir / f"{answer_type}_{model_type}" / "best_model_overall_cv.pth"
        
        if not source_file.exists():
            print(f"  ⚠️  파일 없음: {source_file}")
            continue
        
        # 대상 파일 경로
        dest_file = upload_dir / f"{answer_type}_{model_type}.pth"
        
        # 복사
        shutil.copy(source_file, dest_file)
        
        # 파일 크기 확인
        size_mb = source_file.stat().st_size / (1024 * 1024)
        print(f"  ✅ {dest_file.name} ({size_mb:.2f} MB)")
        copied_files.append(dest_file.name)
    
    # README.md 생성
    readme_content = """# MNIST Fine-tuned Models (Answer Classification)

MNIST base 모델을 Answer_1, Answer_2 데이터셋으로 fine-tuning한 모델들입니다.

## 모델 목록

### Answer_1 모델 (5-class classification)
- `answer_1_efficientnet.pth` - EfficientNet-B0 fine-tuned
- `answer_1_resnet.pth` - ResNet-18 fine-tuned
- `answer_1_mobilenet.pth` - MobileNetV3-Small fine-tuned

### Answer_2 모델 (5-class classification)
- `answer_2_efficientnet.pth` - EfficientNet-B0 fine-tuned
- `answer_2_resnet.pth` - ResNet-18 fine-tuned
- `answer_2_mobilenet.pth` - MobileNetV3-Small fine-tuned

## 사용 방법

```python
import torch
from kaggle.api.kaggle_api_extended import KaggleApi

# 다운로드
api = KaggleApi()
api.authenticate()
api.dataset_download_files('minyujin03/mnist-finetuned-models', path='./models', unzip=True)

# 모델 로드
checkpoint = torch.load('./models/answer_1_efficientnet.pth')
# checkpoint에는 'model_state_dict', 'best_cv_acc', 'mean_cv_acc' 등이 포함됨
```

## 모델 정보
- **Base Models**: MNIST pre-trained (from `minyujin03/mnist-base-models`)
- **Fine-tuning**: Answer_1, Answer_2 datasets  
- **클래스 수**: 5 (1, 2, 3, 4, 5)
- **Cross Validation**: 5-fold
- **Input Size**: 28x28 (grayscale)
"""
    
    readme_path = upload_dir / "README.md"
    with open(readme_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)
    
    print(f"  ✅ README.md")
    
    print(f"\n{'='*70}")
    print(f"✅ 준비 완료!")
    print(f"📁 업로드 폴더: {upload_dir.absolute()}")
    print(f"📦 파일 개수: {len(copied_files) + 1} (모델 {len(copied_files)}개 + README)")
    print(f"\n💡 다음 단계:")
    print(f"   1. https://www.kaggle.com/datasets 접속")
    print(f"   2. 'New Dataset' 클릭")
    print(f"   3. '{upload_dir}' 폴더의 모든 파일을 드래그 앤 드롭")
    print(f"   4. Title: 'mnist-finetuned-models'")
    print(f"   5. 'Create' 클릭")
    print(f"{'='*70}")

if __name__ == "__main__":
    prepare_finetuned_models()

📦 Fine-tuned 모델 업로드 준비

📋 복사할 모델:
  ✅ answer_1_efficientnet.pth (4.15 MB)
  ✅ answer_1_resnet.pth (42.69 MB)
  ✅ answer_1_mobilenet.pth (3.86 MB)
  ✅ answer_2_efficientnet.pth (4.15 MB)
  ✅ answer_2_resnet.pth (42.69 MB)
  ✅ answer_2_mobilenet.pth (3.86 MB)
  ✅ README.md

✅ 준비 완료!
📁 업로드 폴더: /Users/min-yujin/Gradi_25Fall/ai-service/test/CNN/kaggle_finetuned_upload
📦 파일 개수: 7 (모델 6개 + README)

💡 다음 단계:
   1. https://www.kaggle.com/datasets 접속
   2. 'New Dataset' 클릭
   3. 'kaggle_finetuned_upload' 폴더의 모든 파일을 드래그 앤 드롭
   4. Title: 'mnist-finetuned-models'
   5. 'Create' 클릭
